In [ ]:
import torch
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForCausalLM
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datasets import Dataset, DatasetDict


model_name = "openai-community/gpt2"
FILE_PATH = "/content/guardian_headlines.csv"

def load_csv(file_path: str):
  return Dataset.from_csv(file_path)

ds = load_csv(FILE_PATH)


In [2]:
train_test_ds = ds.train_test_split(test_size=0.1)
train_ds = train_test_ds['train']
test_ds = train_test_ds['test']

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:
all_columns = ds.column_names

In [5]:
all_columns

['Headlines', 'labels']

In [ ]:
def tokenize(batch):
  tokenized_inputs = tokenizer(batch['Headlines'], padding="max_length", truncation=True, return_tensors="pt")
  tokenized_inputs["labels"] = tokenized_inputs["input_ids"].clone()
  return tokenized_inputs
train_tokenized = train_ds.map(tokenize, batched=True, remove_columns=['Headlines', 'labels'])
test_tokenized = test_ds.map(tokenize, batched=True, remove_columns=['Headlines', 'labels'])


Map:   0%|          | 0/16020 [00:00<?, ? examples/s]

Map:   0%|          | 0/1780 [00:00<?, ? examples/s]

In [9]:
train_tokenized

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 16020
})

In [10]:
training_args = TrainingArguments(
    output_dir="headlines-gpt2-headlines",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=3,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,
    fp16=True,

    logging_steps=20,
    label_names=['labels'],


    remove_unused_columns=False,

)


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    processing_class=tokenizer
)



In [12]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.056192,0.052841
2,0.047375,0.051841


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=4006, training_loss=0.07566466979604332, metrics={'train_runtime': 3970.4614, 'train_samples_per_second': 8.07, 'train_steps_per_second': 1.009, 'total_flos': 1.674359341056e+16, 'train_loss': 0.07566466979604332, 'epoch': 2.0})

In [13]:
model.save_pretrained("headlines-gpt2-headlines")
tokenizer.save_pretrained("headlines-gpt2-headlines")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('headlines-gpt2-headlines/tokenizer_config.json',
 'headlines-gpt2-headlines/tokenizer.json')

In [16]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [17]:

import math


def perplexity(loss: float) -> float:
    """Convert a mean cross-entropy loss into perplexity.

    Args:
        loss: Mean per-token cross-entropy in nats.

    Returns:
        ``exp(loss)``, or infinity when the loss is too large to exponentiate.
    """
    try:
        return math.exp(loss)
    except OverflowError:
        return float("inf")

In [27]:
ADAPERTER_MODEL = "/content/headlines-gpt2-headlines"

tokenizer = AutoTokenizer.from_pretrained(
    ADAPERTER_MODEL
)

base_model = AutoModelForCausalLM.from_pretrained(model_name).eval()


tuned_model = AutoModelForCausalLM.from_pretrained(
    ADAPERTER_MODEL
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [28]:
def get_perplexity(model_to_evaluate, dataset):

    eval_trainer = Trainer(
        model=model_to_evaluate,
        args=training_args,
        eval_dataset=dataset
    )
    eval_results = eval_trainer.evaluate()
    loss = eval_results['eval_loss']
    return perplexity(loss)


base_model_perplexity = get_perplexity(base_model, test_tokenized)
print(f"Base Model Perplexity: {base_model_perplexity}")


tuned_model_perplexity = get_perplexity(tuned_model, test_tokenized)
print(f"Tuned Model Perplexity: {tuned_model_perplexity}")

Training Loss,Validation Loss,Epoch
No log,9.698040,0


Base Model Perplexity: 16285.65614945958


Training Loss,Validation Loss,Epoch
No log,0.051841,0


Tuned Model Perplexity: 1.053208063200787
